# Generators - Process Massive Data Without Exploding Your RAM

## The Problem Generators Solve
You have a 50GB log file. You need to process it. What do you do?

__Bad approach__:

In [ ]:
with open('50gb_file.txt', 'r') as f:
    lines = f.readlines()  # Loads ALL 50GB into RAM
    for line in lines:
        process(line)
# Result: Your program crashes. Out of memory.

__Good Approach__:

In [ ]:
with open('50gb_file.txt', 'r') as f:
    for line in f:  # Reads ONE line at a time
        process(line)
# Result: Uses ~1KB of RAM regardless of file size

That's what generators do. They produce values ONE AT A TIME instead of creating entire lists in memory.

## Part 1: Lists vs Generators - The Memory Problem

### List Comprehension (Bad for Large Data)

In [ ]:
# Create a list of 10 million squares
squares = [x**2 for x in range(10_000_000)]

# Memory usage: ~80MB (each int is ~8 bytes)
# All values computed and stored IMMEDIATELY
print(f"First 5: {squares[:5]}")
# Problem: What if you only needed the first 5? You just wasted 80MB.

### Generator Expression (Good for Large Data)

In [ ]:
# Create a generator of 10 million squares
squares = (x**2 for x in range(10_000_000))  # NOTE: () not []

# Memory usage: ~200 bytes (tiny!)
# NO values computed yet - it's LAZY
print(f"Type: {type(squares)}")  # <class 'generator'>

# Values computed ON DEMAND
for i, square in enumerate(squares):
    print(square)
    if i >= 4:  # Only get first 5
        break
# Only computed 5 values, not 10 million

### Key difference:
- [x**2 for x in range(N)] - Creates list, computes ALL values NOW, stores in memory
- (x**2 for x in range(N)) - Creates generator, computes values WHEN ASKED, stores nothing

## Part 2: Creating Generators with yield
A function becomes a generator when it uses yield instead of return.

### Regular Function (Returns Everything at Once)

In [ ]:
def get_numbers():
    results = []
    for i in range(5):
        results.append(i)
    return results

numbers = get_numbers()
print(numbers)  # [0, 1, 2, 3, 4] - all at once

### Generator Function (Yields One at a Time)

In [ ]:
def get_numbers():
    for i in range(5):
        yield i  # Pause here and return value

numbers = get_numbers()
print(numbers)  # <generator object> - not the values yet!
print()

# Get values one by one
print(next(numbers))  # 0
print(next(numbers))  # 1
print(next(numbers))  # 2
print("---")
# Or iterate
for num in get_numbers():
    print(num)  # 0, 1, 2, 3, 4

What yield does:
1. Returns a value
2. PAUSES the function (saves its state)
3. Next time you call it, RESUMES from where it paused

## Part 3: How yield Actually Works (The Execution Model)
This is critical to understand:

In [ ]:
def simple_generator():
    print("Start")
    yield 1
    print("Between 1 and 2")
    yield 2
    print("Between 2 and 3")
    yield 3
    print("End")

gen = simple_generator()
print("Generator created")

print(next(gen))  # Prints "Start", yields 1
print(next(gen))  # Prints "Between 1 and 2", yields 2
print(next(gen))  # Prints "Between 2 and 3", yields 3
# next(gen)       # Prints "End", raises StopIteration

"""
Output:
Generator created
Start
1
Between 1 and 2
2
Between 2 and 3
3
"""

__Key insight__: The function PAUSES at each yield. When you call next(), it RESUMES from the last yield.

## Part 4: Real Data Engineering Example - Reading Large Files

### Bad: Loading Entire File

In [ ]:
def read_file_bad(filename):
    with open(filename) as f:
        return f.readlines()  # Loads ENTIRE file into memory

lines = read_file_bad('huge_log.txt')  # 10GB file = 10GB RAM
for line in lines:
    if 'ERROR' in line:
        print(line)

### Good: Generator Approach

In [ ]:
def read_file(filename):
    with open(filename) as f:
        for line in f:
            yield line.strip()

# Memory usage: ~1 line at a time
for line in read_file('huge_log.txt'):
    if 'ERROR' in line:
        print(line)

### Even better: Generator with filtering

In [ ]:
def read_errors(filename):
    with open(filename) as f:
        for line in f:
            if 'ERROR' in line:
                yield line.strip()

# Only yields ERROR lines, nothing else
for error_line in read_errors('huge_log.txt'):
    print(error_line)

## Part 5: Generator Pipelines (This is Where it Gets Powerful)
You can chain generators together. Each one processes data from the previous one, ONE ITEM AT A TIME.

### Example: ETL Pipeline

In [ ]:
# Create test data
with open('sales.csv', 'w') as f:
    f.write("id,product,amount,region\n")
    f.write("1,Widget,100,North\n")
    f.write("2,Gadget,150,South\n")
    f.write("3,Widget,200,North\n")
    f.write("4,Gadget,120,East\n")
    f.write("5,Widget,180,South\n")

# Stage 1: Read file line by line
def read_file(filename):
    with open(filename) as f:
        for line in f:
            yield line.strip()

# Stage 2: Parse CSV
def parse_csv(lines):
    header = None
    for line in lines:
        if header is None:
            header = line.split(',')
            continue
        values = line.split(',')
        yield dict(zip(header, values))

# Stage 3: Filter records
def filter_product(records, product):
    for record in records:
        if record['product'] == product:
            yield record

# Stage 4: Extract field
def extract_amounts(records):
    for record in records:
        yield int(record['amount'])

# Chain them together!
lines = read_file('sales.csv')
records = parse_csv(lines)
widgets = filter_product(records, 'Widget')
amounts = extract_amounts(widgets)
total = sum(amounts)

print(f"Total Widget sales: ${total}")  # $480

__What just happened?__
- NO intermediate lists created
- Each stage processes ONE item at a time
- Data flows through the pipeline like water through pipes
- Memory usage: CONSTANT (1 item at a time)

__This is how you process TB-scale data on a laptop.__

## Part 6: Generator Expression (Shorthand)
Instead of writing full generator function, you can use a generator expression:

In [ ]:
# Generator function
def squares(n):
    for i in range(n):
        yield i ** 2

# Generator expression (same thing, shorter)
squares = (i**2 for i in range(n))

# Use it the same way
for sq in squares:
    print(sq)

__When to use each:__
- Generator function: Complex logic, multiple yields, need to maintain state
- Generator expression: Simple transformations, one-liners

### Example: Filtering large datasets

In [ ]:
# Read 1 billion numbers, filter evens, sum first 10
numbers = (x for x in range(1_000_000_000))
evens = (x for x in numbers if x % 2 == 0)
first_10_evens = (x for i, x in enumerate(evens) if i < 10)
total = sum(first_10_evens)

# Memory used: ~constant
# Without generators: Would need ~8GB RAM

## Part 7: Infinite Generators
Generators can produce infinite sequences because they're lazy:

In [ ]:
def infinite_counter():
    n = 0
    while True:  # Infinite loop!
        yield n
        n += 1

counter = infinite_counter()
print(next(counter))  # 0
print(next(counter))  # 1
print(next(counter))  # 2
# ... forever

### Fibonacci sequence (infinite):

In [ ]:
def fibonacci():
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b

# Get first 10 Fibonacci numbers
from itertools import islice
fib = fibonacci()
first_10 = list(islice(fib, 10))
print(first_10)  # [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]

## Part 8: Advanced - Generator Methods
Generators have methods you can call:

### 1. send() - Send values INTO a generator

In [ ]:
def accumulator():
    total = 0
    while True:
        value = yield total  # Yield current total, receive new value
        if value is not None:
            total += value

acc = accumulator()
next(acc)  # Prime the generator (get to first yield)

print(acc.send(10))  # Send 10, get total: 10
print(acc.send(20))  # Send 20, get total: 30
print(acc.send(15))  # Send 15, get total: 45

Real use case: Running average

In [ ]:
def running_average():
    total = 0
    count = 0
    while True:
        value = yield total / count if count > 0 else 0
        if value is not None:
            total += value
            count += 1

avg = running_average()
next(avg)  # Prime

print(avg.send(10))   # 10.0
print(avg.send(20))   # 15.0
print(avg.send(30))   # 20.0

### 2. throw() - Throw exception into generator

In [ ]:
def resilient_generator():
    while True:
        try:
            value = yield
            print(f"Received: {value}")
        except ValueError:
            print("Caught ValueError, continuing...")

gen = resilient_generator()
next(gen)

gen.send("hello")
gen.throw(ValueError, "Bad value")  # Generator catches it
gen.send("world")

### 3. close() - Stop the generator

In [ ]:
def generator():
    try:
        while True:
            yield "value"
    finally:
        print("Cleanup: closing resources")

gen = generator()
next(gen)
gen.close()  # Triggers finally block

## Part 9: Real ETL Pipeline Example
This is a production-quality ETL pipeline using generators:

In [ ]:
import csv
import json
from datetime import datetime

# Extract: Read CSV
def extract_csv(filename):
    """Read CSV file line by line"""
    with open(filename, 'r') as f:
        reader = csv.DictReader(f)
        for row in reader:
            yield row

# Transform: Clean and enrich
def transform_records(records):
    """Transform each record"""
    for record in records:
        # Convert types
        record['amount'] = float(record['amount'])
        record['quantity'] = int(record.get('quantity', 1))
        
        # Add derived fields
        record['total'] = record['amount'] * record['quantity']
        record['processed_at'] = datetime.now().isoformat()
        
        # Clean data
        record['product'] = record['product'].strip().upper()
        
        yield record

# Filter: Remove invalid records
def filter_valid(records):
    """Keep only valid records"""
    for record in records:
        if record['amount'] > 0 and record['quantity'] > 0:
            yield record
        else:
            print(f"Skipping invalid record: {record['id']}")

# Aggregate: Group by product
def aggregate_by_product(records):
    """Aggregate sales by product"""
    products = {}
    for record in records:
        product = record['product']
        if product not in products:
            products[product] = {
                'product': product,
                'total_sales': 0,
                'total_quantity': 0,
                'count': 0
            }
        products[product]['total_sales'] += record['total']
        products[product]['total_quantity'] += record['quantity']
        products[product]['count'] += 1
    
    for product_data in products.values():
        yield product_data

# Load: Write to JSON
def load_json(records, filename):
    """Write records to JSON file"""
    records_list = list(records)  # Consume generator
    with open(filename, 'w') as f:
        json.dump(records_list, f, indent=2)
    return len(records_list)

# Build pipeline
def run_etl(input_file, output_file):
    """Run complete ETL pipeline"""
    pipeline = load_json(
        aggregate_by_product(
            filter_valid(
                transform_records(
                    extract_csv(input_file)
                )
            )
        ),
        output_file
    )
    print(f"Processed {pipeline} products")

# Test
with open('sales_data.csv', 'w') as f:
    f.write("id,product,amount,quantity\n")
    f.write("1,widget,10.50,2\n")
    f.write("2,gadget,20.00,1\n")
    f.write("3,widget,10.50,3\n")
    f.write("4,gadget,20.00,2\n")

run_etl('sales_data.csv', 'output.json')

__Why this is good:__
- Each stage is independent and testable
- Memory usage is constant (processes one record at a time)
- Easy to add/remove stages
- Can handle files of any size